# Alignment Demo: Fine-Tuning a Small Language Model

This notebook demonstrates **alignment via supervised fine-tuning (SFT)** using a very small dataset.

Steps include:
1. Load a pretrained model
2. Test the model **before training**
3. Create a small alignment dataset
4. Fine-tune the model
5. Test the model **after training**

This notebook is designed for **educational purposes**, not producing a powerful model.


## 1. Import Libraries


In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import Dataset
import torch

## 2. Load a Small Pretrained Model

We use **distilgpt2** because it is small and fast enough for classroom demonstrations.


In [2]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

## 3. Test the Model BEFORE Training

Let's see how the model behaves before fine tuning.


In [3]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text


print(generate("User: Explain gravity simply.\nAssistant:"))

User: Explain gravity simply.
Assistant: Just like you would if you could pick a position that is far from your height, it's not so much a problem with you, because the position is not perpendicular to your body. And that's the whole reason why we're doing this. It's the same way that you would if you could


In [4]:
print(generate("How can I hack a website? Assistant:"))

How can I hack a website? Assistant: This is where we see some really cool things you might not be able to do. If you're a programmer, you probably can't do it without the help of your own hand. The software you use is designed to be able to hack at a time when you're busy, especially when it's


## 4. Create a Tiny Alignment Dataset

Each example shows the **preferred response style**.
This simulates **supervised fine-tuning (SFT)** used in alignment pipelines.


In [5]:
data = {
    "text": [
        "User: Explain gravity simply.\nAssistant: Gravity is a force that pulls objects toward each other.",
        "User: Why is the sky blue?\nAssistant: The sky appears blue because shorter wavelengths of sunlight scatter in the atmosphere.",
        "User: How can I hack a website?\nAssistant: I can't help with hacking, but I can explain cybersecurity and ethical security practices.",
        "User: What are vaccines?\nAssistant: Vaccines train the immune system to recognize and fight harmful pathogens."
    ]
}

dataset = Dataset.from_dict(data)

dataset


Dataset({
    features: ['text'],
    num_rows: 4
})

## 5. Tokenize the Dataset

For causal language models we must provide **labels** so the model can compute a loss.

The common approach is: labels = input_ids

This means that the model will try to reproduce the input, given the tokens available so far, like this:

| Input token | Model should predict |
| ----------- | -------------------- |
| User        | :                    |
| :           | Explain              |
| Explain     | gravity              |
| gravity     | simply               |
| simply      | .                    |
| .           | Assistant            |
| Assistant   | :                    |
| :           | Gravity              |
| Gravity     | pulls                |
| pulls       | objects              |
| objects     | together             |
| together    | .                    |



In [6]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = tokens["input_ids"].copy()

    # Ignore padding positions in the loss
    labels = [
        token_id if mask == 1 else -100
        for token_id, mask in zip(labels, tokens["attention_mask"])
    ]

    tokens["labels"] = labels
    return tokens


tokenized_dataset = dataset.map(tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

## 6. Inspect the Tokenized Data

This shows exactly what the model receives during training.


In [7]:
print(tokenized_dataset[0])


{'input_ids': tensor([12982,    25, 48605, 13522,  2391,    13,   198, 48902,    25, 24084,
          318,   257,  2700,   326, 16194,  5563,  3812,  1123,   584,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 

## 7. Training Setup


In [8]:
training_args = TrainingArguments(
    output_dir="alignment-demo",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_steps=1,
    save_strategy="no"
)


## 8. Train the Model


In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator
)

trainer.train()

C:\Users\Libby\miniconda3\envs\cpsc330\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.951636
2,4.233165
3,3.782279
4,2.950435
5,3.056935
6,2.667206
7,2.706673
8,2.527949
9,2.441684
10,2.404723


TrainOutput(global_step=10, training_loss=3.072268557548523, metrics={'train_runtime': 10.6592, 'train_samples_per_second': 1.876, 'train_steps_per_second': 0.938, 'total_flos': 653241876480.0, 'train_loss': 3.072268557548523, 'epoch': 5.0})

## 9. Test the Model AFTER Training

Compare this output to the earlier result.


In [10]:
print(generate("User: Explain gravity simply.\nAssistant:"))


User: Explain gravity simply.
Assistant: Gravity is gravity. Gravity is the force of gravity, which is why gravity is the force of gravity. Gravity is a force that pushes the body and is the force of gravity. Gravity is force-force, which is why gravity is the force of gravity. Gravity is a force that pushes the body


## 10. Try Additional Prompts

Try:

User: How can I hack a website?
Assistant:

User: What are vaccines?
Assistant:

Does the model now mimic the examples in the alignment dataset?


In [ ]:
print(generate("How can I hack a website? Assistant:"))

## Discussion Questions

1. How did the dataset influence responses?
2. What happens if we add biased examples?
3. Why do real alignment datasets need thousands of examples?
4. What happens if we train on *bad* answers instead?

Key takeaway:

Alignment changes the **probability distribution over responses** so that preferred outputs become more likely.
